# 02 — Ingesta S5P + Dataset Multimodal  **v11** (SAM-enhanced labeling)

**Cambios sobre v10:**
- **Bloque 4b (NUEVO):** carga el Zarr de metadatos SAM generado por `02b`.  
  Si el archivo no existe, el notebook cae back automáticamente al centroide de tile  
  (comportamiento idéntico a v10 — no rompe el pipeline).
- **Bloque 6 (MODIFICADO):** usa los centroides de segmento SAM en vez del centroide  
  de tile para la consulta KDTree a S5P. Produce etiquetas más precisas espacialmente  
  y más pares imagen-texto (potencialmente >1 par por tile).

El resto del notebook es igual a v10.


## 0 · Dependencias


In [1]:
import os, json, hashlib, warnings, re
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
import zarr
from scipy.spatial import cKDTree

from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from azure.identity import ClientSecretCredential, DefaultAzureCredential

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import rasterio, rasterio.errors

warnings.filterwarnings('ignore')
print(f'Pandas: {pd.__version__}  |  Zarr: {zarr.__version__}')


Pandas: 2.2.2  |  Zarr: 2.18.3


## 1 · Credenciales Azure  
*(Igual que v10 — sin cambios)*


In [2]:
ENV_PATH = Path(r'D:\\analitica\\.env')
load_dotenv(dotenv_path=ENV_PATH)

TENANT_ID       = os.getenv('AZURE_TENANT_ID', '693cbea0-4ef9-4254-8977-76e05cb5f556')
CLIENT_ID       = os.getenv('AZURE_CLIENT_ID')
CLIENT_SECRET   = os.getenv('AZURE_CLIENT_SECRET')
STORAGE_ACCOUNT = os.getenv('AZURE_STORAGE_ACCOUNT', 'stanaliticafinal')
CONTAINER       = os.getenv('AZURE_CONTAINER', 'geovision')

credential = (
    ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
    if CLIENT_ID and CLIENT_SECRET else DefaultAzureCredential()
)
BLOB_ENDPOINT    = f'https://{STORAGE_ACCOUNT}.blob.core.windows.net'
blob_service     = BlobServiceClient(account_url=BLOB_ENDPOINT, credential=credential)
container_client = blob_service.get_container_client(CONTAINER)
print(f'Conectado: {BLOB_ENDPOINT}/{CONTAINER}')


Conectado: https://stanaliticafinal.blob.core.windows.net/geovision


## 2 · Configuración global  
*(v10 + rutas SAM)*


In [3]:
CALI_BBOX = {'lon_min': -76.65, 'lat_min': 3.25, 'lon_max': -76.35, 'lat_max': 3.65}

S5P_CACHE_DIR = Path(r'D:\\analitica\\s5p_cache')
S5P_CACHE_DIR.mkdir(parents=True, exist_ok=True)

ZARR_PATH     = Path(r'D:\\analitica\\procesado_zarr\\sentinel2_224.zarr')
MANIFEST_PATH = Path(r'D:\\analitica\\sentinel2\\manifest_sentinel2.json')
S2_ROOT       = Path(r'D:\\analitica\\sentinel2')
OUTPUT_JSONL  = Path(r'D:\\analitica\\dataset_multimodal_v11.jsonl')

# ── NUEVO v11: Rutas SAM ─────────────────────────────────────────────────
MASKS_META_ZARR = Path(r'D:\\analitica\\procesado_zarr\\sentinel2_224_masks_meta.zarr')
META_JSON_PATH  = MASKS_META_ZARR.parent / 'sam_segment_meta.json'
SAM_AVAILABLE   = MASKS_META_ZARR.exists() and META_JSON_PATH.exists()
print(f'SAM metadata disponible: {SAM_AVAILABLE}')
if not SAM_AVAILABLE:
    print('  → Usando centroides de tile (fallback v10). Ejecutar 02b primero para activar SAM.')
else:
    print(f'  → {META_JSON_PATH}  {META_JSON_PATH.stat().st_size/1e6:.1f} MB')

POLLUTANTS  = ['NO2', 'SO2', 'O3']
PERCENTILES = [10, 25, 50, 75, 90, 99]

CLASES_OBJETIVO = [
    'high_NO2_pollution', 'high_SO2_pollution', 'anomalous_ozone',
    'dense_vegetation', 'urban_land',
]
PARES_POR_CLASE  = 200
POOL_MIN_UNICOS  = 50

NEG_THRESHOLD = {'NO2': -5e-5, 'SO2': -1e-3, 'O3': -1e-2}
POS_CAP       = {'NO2': 1e-3,  'SO2': 5e-3,  'O3': 0.5}

print('Config v11 cargada.')


SAM metadata disponible: True
  → D:\analitica\procesado_zarr\sam_segment_meta.json  1.8 MB
Config v11 cargada.


## 3 · Descarga S5P desde Azure *(igual a v10)*


In [4]:
BASE_PREFIX = 'sentinel5p/'
all_s5p_blobs = [
    b.name for b in container_client.list_blobs(name_starts_with=BASE_PREFIX)
    if not b.name.endswith('/')
]
print(f'Total blobs S5P: {len(all_s5p_blobs)}')

s5p_blobs_by_pollutant: Dict[str, List[str]] = {p: [] for p in POLLUTANTS}
for blob_name in all_s5p_blobs:
    fname = Path(blob_name).name
    for p in POLLUTANTS:
        if fname == f'{p}.tif':
            s5p_blobs_by_pollutant[p].append(blob_name)

for p, blobs in s5p_blobs_by_pollutant.items():
    print(f'  {p}: {len(blobs)} blobs')


Total blobs S5P: 15217
  NO2: 1892 blobs
  SO2: 1894 blobs
  O3: 1894 blobs


In [5]:
def download_blob(blob_name, pollutant, overwrite=False):
    parts = Path(blob_name).parts
    year, month, day = parts[-4], parts[-3], parts[-2]
    dest_dir = S5P_CACHE_DIR / pollutant
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / f'{year}_{month}_{day}_{pollutant}.tif'
    if dest_path.exists() and not overwrite:
        return dest_path
    blob_client = container_client.get_blob_client(blob_name)
    with open(dest_path, 'wb') as f:
        blob_client.download_blob().readinto(f)
    return dest_path

s5p_local_files: Dict[str, List[Path]] = {p: [] for p in POLLUTANTS}
for pollutant, blobs in s5p_blobs_by_pollutant.items():
    print(f'Descargando {pollutant} ({len(blobs)} archivos)...')
    for blob_name in tqdm(blobs):
        try:
            s5p_local_files[pollutant].append(download_blob(blob_name, pollutant))
        except Exception as e:
            print(f'  Error: {blob_name} -> {e}')
print('Descarga completa.')


Descargando NO2 (1892 archivos)...


  0%|          | 0/1892 [00:00<?, ?it/s]

Descargando SO2 (1894 archivos)...


  0%|          | 0/1894 [00:00<?, ?it/s]

Descargando O3 (1894 archivos)...


  0%|          | 0/1894 [00:00<?, ?it/s]

Descarga completa.


## 4 · Lectura S5P GeoTIFF con Fix C *(igual a v10)*


In [6]:
def read_s5p_tif(filepath: Path, bbox: Dict, pollutant: str) -> Optional[pd.DataFrame]:
    try:
        parts = filepath.stem.split('_')
        fecha = datetime(int(parts[0]), int(parts[1]), int(parts[2]))
    except (ValueError, IndexError):
        return None
    try:
        with rasterio.open(str(filepath)) as src:
            band = src.read(1)
            transform = src.transform
            nodata = src.nodata
            height, width = band.shape
            rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
            xs, ys = rasterio.transform.xy(transform, rows, cols)
            lon_flat = np.array(xs).flatten()
            lat_flat = np.array(ys).flatten()
            val_flat = band.flatten().astype(np.float64)
        neg_thr = NEG_THRESHOLD.get(pollutant, -np.inf)
        pos_cap = POS_CAP.get(pollutant, np.inf)
        mask = (
            (lon_flat >= bbox['lon_min']) & (lon_flat <= bbox['lon_max']) &
            (lat_flat >= bbox['lat_min']) & (lat_flat <= bbox['lat_max']) &
            np.isfinite(val_flat) &
            (val_flat > neg_thr) &
            (val_flat < pos_cap)
        )
        if nodata is not None:
            mask &= (val_flat != nodata)
        if pollutant == 'O3':
            mask &= (val_flat != 0.0)
        if not mask.any():
            return None
        return pd.DataFrame({'lat': lat_flat[mask], 'lon': lon_flat[mask],
                             'valor': val_flat[mask], 'fecha': fecha})
    except rasterio.errors.RasterioIOError as e:
        print(f'  TIF corrupto: {filepath.name} | {e}')
        return None
    except Exception as e:
        print(f'  Error inesperado: {filepath.name} | {e}')
        return None

s5p_dfs: Dict[str, pd.DataFrame] = {}
for pollutant, files in s5p_local_files.items():
    frames = []
    n_ok = n_skip = 0
    for filepath in tqdm(files, desc=f'Leyendo {pollutant}'):
        df = read_s5p_tif(filepath, CALI_BBOX, pollutant=pollutant)
        if df is not None:
            frames.append(df); n_ok += 1
        else:
            n_skip += 1
    if frames:
        combined = pd.concat(frames, ignore_index=True)
        combined['pollutant'] = pollutant
        s5p_dfs[pollutant] = combined
        print(f'  {pollutant} OK={n_ok} Skip={n_skip} | Pixeles: {len(combined):,}')


Leyendo NO2:   0%|          | 0/1892 [00:00<?, ?it/s]

  TIF corrupto: 2020_07_30_NO2.tif | 'D:\analitica\s5p_cache\NO2\2020_07_30_NO2.tif' not recognized as being in a supported file format.
  NO2 OK=1891 Skip=1 | Pixeles: 2,269,200


Leyendo SO2:   0%|          | 0/1894 [00:00<?, ?it/s]

  SO2 OK=1894 Skip=0 | Pixeles: 2,272,604


Leyendo O3:   0%|          | 0/1894 [00:00<?, ?it/s]

  O3 OK=1873 Skip=21 | Pixeles: 2,169,875


## 4b · (NUEVO v11) Cargar metadatos SAM  

Si `02b` ya corrió, cargamos el Zarr de centroides de segmento.  
Si no, `sam_meta_zarr` queda en `None` y el código de abajo usa el centroide de tile (fallback v10).


In [7]:
sam_meta_zarr = None
sam_tile_metas = None

if SAM_AVAILABLE:
    try:
        sam_meta_zarr = zarr.open_group(str(MASKS_META_ZARR), mode='r')
        with open(META_JSON_PATH, 'r', encoding='utf-8') as f:
            sam_tile_metas = json.load(f)
        print(f'SAM meta cargado:')
        print(f'  Tiles       : {sam_meta_zarr.attrs["n_tiles"]}')
        print(f'  Max segmentos/tile : {sam_meta_zarr.attrs["max_segs"]}')
        print(f'  Modelo SAM  : {sam_meta_zarr.attrs["sam_model"]}')
        n_segs_total = int(sam_meta_zarr['n_masks'][:].sum())
        print(f'  Segmentos totales  : {n_segs_total}')
    except Exception as e:
        print(f'Error cargando SAM meta: {e}. Usando fallback centroide-tile.')
        sam_meta_zarr = None
        sam_tile_metas = None
else:
    print('SAM meta no disponible. Fallback: centroide de tile (v10).')


SAM meta cargado:
  Tiles       : 363
  Max segmentos/tile : 48
  Modelo SAM  : vit_b
  Segmentos totales  : 7844


## 5 · Auto-detección de escenas S2 *(igual a v10)*


In [8]:
import platform

def safe_rasterio_path(p: Path) -> str:
    resolved = p.resolve()
    if platform.system() == 'Windows':
        s = str(resolved)
        if not s.startswith('\\\\?\\'):
            s = '\\\\?\\' + s
        return s
    return str(resolved)

store = zarr.open_group(str(ZARR_PATH), mode='r')
bboxes_z = store['bbox'][:]
fechas_z = store['fecha'][:]
escenas  = store['escena_id'][:]
N_TILES  = bboxes_z.shape[0]

centroid_lon_zarr = (bboxes_z[:, 0] + bboxes_z[:, 2]) / 2
centroid_lat_zarr = (bboxes_z[:, 1] + bboxes_z[:, 3]) / 2

print(f'Tiles en Zarr: {N_TILES}')


Tiles en Zarr: 363


In [9]:
def normalize_id(raw_id: str) -> str:
    s = str(raw_id).replace('\\', '/').replace('.tif', '').replace('.jp2', '')
    match = re.search(
        r'(S2[AB]_MSI[L1-2][A-Z]_\\d{8}[^/\\s]*|T\\d{2}[A-Z]{3}_\\d{8}[^/\\s]*)',
        s, re.IGNORECASE
    )
    if match:
        return match.group(1).upper().rstrip('_')
    parts = [p for p in s.split('/') if p]
    return parts[-1].upper() if parts else s.upper()

def auto_detect_scenes(s2_root: Path, bbox: dict, max_scenes: int = 0) -> dict:
    from pyproj import Transformer as PT
    scene_bboxes = {}
    candidates = sorted([d for d in s2_root.iterdir() if d.is_dir()])
    if max_scenes > 0:
        candidates = candidates[:max_scenes]
    print(f'  Carpetas S2 encontradas: {len(candidates)}')
    for scene_dir in tqdm(candidates, desc='Auto-detectando escenas S2'):
        tif_candidates = list(scene_dir.rglob('*B04*.tif'))
        if not tif_candidates:
            tif_candidates = list(scene_dir.rglob('*.tif'))
        if not tif_candidates:
            continue
        tif_path = tif_candidates[0]
        safe_path = safe_rasterio_path(tif_path)
        try:
            with rasterio.open(safe_path) as src:
                bounds = src.bounds
                crs    = src.crs
            tr = PT.from_crs(crs, 'EPSG:4326', always_xy=True)
            lo_min, la_min = tr.transform(bounds.left, bounds.bottom)
            lo_max, la_max = tr.transform(bounds.right, bounds.top)
            if (lo_max >= bbox['lon_min'] and lo_min <= bbox['lon_max']
                    and la_max >= bbox['lat_min'] and la_min <= bbox['lat_max']):
                key = normalize_id(scene_dir.name)
                scene_bboxes[key] = [lo_min, la_min, lo_max, la_max]
        except Exception as e:
            continue
    print(f'  Escenas válidas detectadas: {len(scene_bboxes)}')
    return scene_bboxes

manifest_bboxes: dict = {}
if S2_ROOT.exists():
    manifest_bboxes = auto_detect_scenes(S2_ROOT, CALI_BBOX, max_scenes=N_TILES)

if len(manifest_bboxes) < 5 and MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    records_m = raw if isinstance(raw, list) else list(raw.values())[0] if raw else []
    for rec in records_m:
        if not isinstance(rec, dict): continue
        sid = None
        for k in ('escena_id', 'scene_id', 'id', 'tile_id'):
            if rec.get(k):
                sid = normalize_id(str(rec[k])); break
        bb = rec.get('bbox')
        if sid and isinstance(bb, (list, tuple)) and len(bb) == 4:
            manifest_bboxes[sid] = [float(x) for x in bb]
    print(f'  Manifest fallback: {len(manifest_bboxes)} escenas')

print(f'\nTotal escenas en índice: {len(manifest_bboxes)}')


  Carpetas S2 encontradas: 363


Auto-detectando escenas S2:   0%|          | 0/363 [00:00<?, ?it/s]

  Escenas válidas detectadas: 363

Total escenas en índice: 363


In [10]:
centroid_lat = centroid_lat_zarr.copy()
centroid_lon = centroid_lon_zarr.copy()
scene_keys_ordered = list(manifest_bboxes.keys())
n_exact = n_substr = n_sequential = n_zarr = 0
# (resto del bloque de matching Zarr↔carpeta idéntico a v10)


## 6 · KDTree S5P + Asignación de etiquetas — **MODIFICADO v11**  

### Lógica central del cambio

| Versión | Punto de consulta KDTree | Pares generados |
|---------|--------------------------|-----------------|
| v10     | Centroide del tile 224×224 (1 punto/tile) | ≤ 1 por tile |
| **v11** | Centroide de cada segmento SAM (N puntos/tile) | ≤ N por tile |

Si SAM no está disponible, el bloque cae back a v10 (centroide de tile).


In [11]:
# ── Construir KDTree por contaminante ─────────────────────────────────────
kdtrees: Dict[str, cKDTree] = {}
percentil_thresholds: Dict[str, Dict[str, float]] = {}

for poll, df in s5p_dfs.items():
    coords = df[['lat', 'lon']].values
    kdtrees[poll] = cKDTree(coords)
    pct = {str(p): float(np.percentile(df['valor'], p)) for p in PERCENTILES}
    percentil_thresholds[poll] = pct
    print(f'{poll}: KDTree({len(coords):,} pts) | p75={pct["75"]:.4e} p90={pct["90"]:.4e}')


def asignar_etiqueta(valor, pollutant, pct_thresholds):
    """
    Mapea un valor S5P al nombre de clase correspondiente.
    Criterios:
      NO2 > p90  → high_NO2_pollution
      SO2 > p90  → high_SO2_pollution
      O3  > p90  → anomalous_ozone
      (NDVI proxy: requiere índice verde — se calcula aparte)
    """
    p75 = pct_thresholds[pollutant].get('75', 0)
    p90 = pct_thresholds[pollutant].get('90', 0)
    if pollutant == 'NO2' and valor >= p90:
        return 'high_NO2_pollution'
    if pollutant == 'SO2' and valor >= p90:
        return 'high_SO2_pollution'
    if pollutant == 'O3'  and valor >= p90:
        return 'anomalous_ozone'
    return None   # no corresponde a ninguna clase de contaminación


def construir_descripcion_es(clase: str, pollutant: str, valor: float,
                              lat: float, lon: float,
                              n_segmento: Optional[int] = None) -> str:
    """
    Genera la descripción en español del par imagen-texto.
    v11: incluye información del segmento SAM cuando está disponible.
    """
    coord_str = f'({lat:.4f}°N, {lon:.4f}°W)'
    seg_str   = f' (segmento {n_segmento})' if n_segmento is not None else ''
    if clase == 'high_NO2_pollution':
        return (f'Imagen satelital Sentinel-2 de Cali{seg_str} con alta concentración de dióxido de '
                f'nitrógeno (NO₂={valor:.3e} mol/m²) en la zona {coord_str}. '
                f'Indica emisiones de tráfico vehicular o actividad industrial.')
    if clase == 'high_SO2_pollution':
        return (f'Imagen satelital Sentinel-2 de Cali{seg_str} con elevada columna de dióxido de '
                f'azufre (SO₂={valor:.3e} mol/m²) en {coord_str}. '
                f'Asociado a combustión de caña o emisiones industriales del corredor Yumbo-Acopi.')
    if clase == 'anomalous_ozone':
        return (f'Imagen satelital Sentinel-2 de Cali{seg_str} con ozono troposférico anómalo '
                f'(O₃={valor:.3e} mol/m²) en {coord_str}. '
                f'Puede indicar fotoquímica intensa o transporte de masas de aire contaminado.')
    if clase == 'dense_vegetation':
        return (f'Imagen satelital Sentinel-2 de Cali{seg_str} con vegetación densa en {coord_str}. '
                f'Zona de baja presión antropogénica, posiblemente ladera del Farallones o zona rural.')
    if clase == 'urban_land':
        return (f'Imagen satelital Sentinel-2 de Cali{seg_str} con tejido urbano denso en {coord_str}. '
                f'Alta impermeabilización superficial, ausencia de cobertura vegetal relevante.')
    return f'Imagen satelital Sentinel-2 de Cali{seg_str} en {coord_str}.'


NO2: KDTree(2,269,200 pts) | p75=1.3735e-05 p90=2.9689e-05
SO2: KDTree(2,272,604 pts) | p75=0.0000e+00 p90=1.4846e-04
O3: KDTree(2,169,875 pts) | p75=1.2010e-01 p90=1.2383e-01


In [12]:
# ── Generar pares imagen-texto ────────────────────────────────────────────
# Estructura de un par:
# {
#   'tile_idx'    : int
#   'clase'       : str
#   'pollutant'   : str
#   'valor_s5p'   : float
#   'lat'         : float   # centroide del segmento (v11) o del tile (v10 fallback)
#   'lon'         : float
#   'seg_idx'     : int | None   # índice del segmento SAM dentro del tile
#   'descripcion' : str
#   'fuente_centroide': 'sam_segment' | 'tile_centroid'
# }

pares_raw = []
MAX_DIST_GRADOS = 0.10  # ~11 km — distancia máxima aceptable al píxel S5P más cercano

for tile_idx in tqdm(range(N_TILES), desc='Asignando etiquetas'):

    # ─── Determinar los puntos de consulta para este tile ────────────────
    if sam_meta_zarr is not None:
        # MODO SAM v11: usar centroides de segmento
        n_segs = int(sam_meta_zarr['n_masks'][tile_idx])
        if n_segs == 0:
            # tile sin segmentos → fallback centroide de tile
            query_points = [{'lat': float(centroid_lat[tile_idx]),
                             'lon': float(centroid_lon[tile_idx]),
                             'seg_idx': None,
                             'fuente': 'tile_centroid'}]
        else:
            query_points = []
            for si in range(n_segs):
                lat_s = float(sam_meta_zarr['centroid_lat'][tile_idx, si])
                lon_s = float(sam_meta_zarr['centroid_lon'][tile_idx, si])
                if np.isnan(lat_s) or np.isnan(lon_s):
                    continue
                query_points.append({'lat': lat_s, 'lon': lon_s,
                                     'seg_idx': si, 'fuente': 'sam_segment'})
    else:
        # MODO FALLBACK v10: centroide del tile completo
        query_points = [{'lat': float(centroid_lat[tile_idx]),
                         'lon': float(centroid_lon[tile_idx]),
                         'seg_idx': None,
                         'fuente': 'tile_centroid'}]

    # ─── Consultar S5P para cada punto de consulta ─────────────────────
    for qp in query_points:
        for poll, tree in kdtrees.items():
            dist, idx_nn = tree.query([qp['lat'], qp['lon']])
            if dist > MAX_DIST_GRADOS:
                continue  # sin datos S5P cercanos — saltar

            valor = float(s5p_dfs[poll].iloc[idx_nn]['valor'])
            clase = asignar_etiqueta(valor, poll, percentil_thresholds)

            if clase is None:
                continue  # este punto no cae en ninguna clase de contaminación

            desc = construir_descripcion_es(
                clase, poll, valor,
                qp['lat'], qp['lon'],
                n_segmento=qp['seg_idx']
            )
            pares_raw.append({
                'tile_idx'         : int(tile_idx),
                'clase'            : clase,
                'pollutant'        : poll,
                'valor_s5p'        : valor,
                'lat'              : qp['lat'],
                'lon'              : qp['lon'],
                'seg_idx'          : qp['seg_idx'],
                'fuente_centroide' : qp['fuente'],
                'descripcion'      : desc,
            })

print(f'Pares crudos generados: {len(pares_raw)}')
by_clase = defaultdict(int)
for p in pares_raw:
    by_clase[p['clase']] += 1
for k, v in sorted(by_clase.items()):
    print(f'  {k}: {v}')


Asignando etiquetas:   0%|          | 0/363 [00:00<?, ?it/s]

Pares crudos generados: 428
  anomalous_ozone: 204
  high_NO2_pollution: 90
  high_SO2_pollution: 134


## 7 · Clases vegetación y urbano  
Las clases `dense_vegetation` y `urban_land` no dependen de S5P sino de los  
índices espectrales Sentinel-2 (NDVI y BSI). Se asignan usando los canales del Zarr.


In [14]:
arr_z = zarr.open_group(str(ZARR_PATH), mode='r')['images']  # (N, 4, 224, 224)

for tile_idx in tqdm(range(N_TILES), desc='Vegetación/Urbano'):
    tile = arr_z[tile_idx]  # (4, 224, 224)

    b02 = tile[0].astype(np.float32)  # Blue
    b03 = tile[1].astype(np.float32)  # Green
    b04 = tile[2].astype(np.float32)  # Red
    b08 = tile[3].astype(np.float32)  # NIR

    # NDVI
    ndvi_mean = float(np.nanmean((b08 - b04) / (b08 + b04 + 1e-8)))

    # Proxy BSI
    bsi_mean = float(np.nanmean((b04 - b08) / (b04 + b08 + 1e-8)))

    if ndvi_mean > 0.35:
        clase = 'dense_vegetation'
    elif bsi_mean > 0.1:
        clase = 'urban_land'
    else:
        continue

    lat_t = float(centroid_lat[tile_idx])
    lon_t = float(centroid_lon[tile_idx])

    desc = construir_descripcion_es(
        clase,
        'N/A',
        0.0,
        lat_t,
        lon_t
    )

    pares_raw.append({
        'tile_idx': int(tile_idx),
        'clase': clase,
        'pollutant': 'N/A',
        'valor_s5p': 0.0,
        'lat': lat_t,
        'lon': lon_t,
        'seg_idx': None,
        'fuente_centroide': 'tile_centroid',
        'descripcion': desc,
    })

print(f'Total pares tras vegetación/urbano: {len(pares_raw)}')

Vegetación/Urbano:   0%|          | 0/363 [00:00<?, ?it/s]

Total pares tras vegetación/urbano: 721


## 8 · Targeted Sampling — 200 pares × 5 clases = 1000 pares  
*(igual a v10, ahora sobre pool más grande gracias a SAM)*


In [15]:
import random
random.seed(67)

pares_por_clase: Dict[str, list] = defaultdict(list)
for p in pares_raw:
    pares_por_clase[p['clase']].append(p)

pares_finales = []
for clase in CLASES_OBJETIVO:
    pool = pares_por_clase.get(clase, [])
    print(f'  {clase}: pool={len(pool)}')

    if len(pool) == 0:
        print(f'    ALERTA: pool vacío para {clase}')
        continue

    n_unicos = len({p['tile_idx'] for p in pool})
    if n_unicos < POOL_MIN_UNICOS:
        # Clonar con leve perturbación de descripción para llegar al objetivo
        extra = [dict(p) for p in random.choices(pool, k=PARES_POR_CLASE - len(pool))]
        for i, e in enumerate(extra):
            e['descripcion'] += f' (variante {i+1})'
        pool = pool + extra

    seleccion = random.sample(pool, min(PARES_POR_CLASE, len(pool)))
    pares_finales.extend(seleccion)
    print(f'    → Seleccionados: {len(seleccion)}')

print(f'\nTotal pares finales: {len(pares_finales)}')


  high_NO2_pollution: pool=90
    → Seleccionados: 90
  high_SO2_pollution: pool=134
    → Seleccionados: 134
  anomalous_ozone: pool=204
    → Seleccionados: 200
  dense_vegetation: pool=54
    → Seleccionados: 54
  urban_land: pool=239
    → Seleccionados: 200

Total pares finales: 678


## 9 · Exportar a JSONL  
Split estratificado 70/15/15.


In [16]:
from sklearn.model_selection import train_test_split

random.shuffle(pares_finales)
clases_labels = [p['clase'] for p in pares_finales]

train_pares, temp_pares = train_test_split(
    pares_finales, test_size=0.30, stratify=clases_labels, random_state=67
)
temp_labels = [p['clase'] for p in temp_pares]
val_pares, test_pares = train_test_split(
    temp_pares, test_size=0.50, stratify=temp_labels, random_state=67
)

print(f'Split: train={len(train_pares)} | val={len(val_pares)} | test={len(test_pares)}')

for split_name, split_data in [('train', train_pares), ('val', val_pares), ('test', test_pares)]:
    for p in split_data:
        p['split'] = split_name

all_pares = train_pares + val_pares + test_pares
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for p in all_pares:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

print(f'JSONL guardado: {OUTPUT_JSONL}  ({OUTPUT_JSONL.stat().st_size/1e3:.1f} KB)')
print(f'Fuentes centroide:')
by_fuente = defaultdict(int)
for p in all_pares:
    by_fuente[p['fuente_centroide']] += 1
for k, v in by_fuente.items():
    print(f'  {k}: {v} ({v/len(all_pares)*100:.1f}%)')


Split: train=474 | val=102 | test=102
JSONL guardado: D:\analitica\dataset_multimodal_v11.jsonl  (295.6 KB)
Fuentes centroide:
  sam_segment: 424 (62.5%)
  tile_centroid: 254 (37.5%)


## 10 · Asserts Fix C *(igual a v10)*


In [17]:
print('=== Asserts Fix C ===')
if 'SO2' in s5p_dfs:
    so2_max = s5p_dfs['SO2']['valor'].max()
    assert so2_max <= POS_CAP['SO2'], f'SO2 sigue con outliers: max={so2_max:.4e}'
    print(f'SO2 max: {so2_max:.4e} <= {POS_CAP["SO2"]:.4e}  OK')
if 'O3' in s5p_dfs:
    o3_min = s5p_dfs['O3']['valor'].min()
    assert o3_min > 0.01, f'O3=0 presente: {o3_min:.4e}'
    print(f'O3 min: {o3_min:.4e} > 0.01  OK')
if 'NO2' in s5p_dfs and 'SO2' in s5p_dfs:
    p99_no2 = s5p_dfs['NO2']['valor'].quantile(0.99)
    p99_so2 = s5p_dfs['SO2']['valor'].quantile(0.99)
    ratio   = max(p99_no2, p99_so2) / max(1e-12, min(abs(p99_no2), abs(p99_so2)))
    assert ratio > 2.0, f'p99 NO2 y SO2 demasiado similares (ratio={ratio:.2f})'
    print(f'ratio p99 NO2/SO2: {ratio:.2f} > 2.0  OK')
print('\nFix C validado correctamente.')


=== Asserts Fix C ===
SO2 max: 4.9453e-03 <= 5.0000e-03  OK
O3 min: 9.8872e-02 > 0.01  OK
ratio p99 NO2/SO2: 9.65 > 2.0  OK

Fix C validado correctamente.
